# Лекция 22. SQL: подзапросы, CTE, оконные функции

Практика после Лекции 21 на наборе данных «Платежи».

Правило: оборот считается только по строкам со `status = 'success'`.
Значения `declined` и `reversed` в оборот не входят.

Оформление ответа: под каждой задачей напишите запрос, затем короткий вывод
в ячейке markdown (что показывает результат).

In [1]:
import os
from pathlib import Path

import pandas as pd
import psycopg2

# Параметры подключения — из переменных окружения или файла .env (см. .env.example в корне)
def _load_dotenv():
    env_path = Path.cwd()
    for base in [env_path, *env_path.parents]:
        dotenv = base / ".env"
        if dotenv.exists():
            for line in dotenv.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, _, value = line.partition("=")
                    os.environ.setdefault(key.strip(), value.strip())
            break

_load_dotenv()

conn = psycopg2.connect(
    host=os.environ["DB_HOST"],
    port=int(os.environ.get("DB_PORT", "5432")),
    dbname=os.environ["DB_NAME"],
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    sslmode=os.environ.get("DB_SSLMODE", "require"),
)

def sql(query):
    """Выполнить SQL и вернуть результат как DataFrame."""
    return pd.read_sql(query, conn)

print("Подключено")

Подключено


## Набор данных

| Таблица | Содержимое | Ключи |
|---|---|---|
| `customers` | 120 клиентов: имя, город, пол, даты | `customer_id` |
| `cards` | 179 карт: тип, статус, владелец | `card_id`, `customer_id` |
| `merchants` | 40 мерчантов: название, категория, город | `merchant_id` |
| `transactions` | 4 671 платёж: сумма, время, статус, канал | `txn_id`, `card_id`, `merchant_id` |

Связи: `transactions.card_id` ссылается на `cards`, `cards.customer_id` на
`customers`, `transactions.merchant_id` на `merchants`.

Сумма платежа лежит в `transactions`, категория у мерчанта. Чтобы от платежа
дойти до клиента, нужно два JOIN: транзакция, карта, клиент.

In [2]:
display(sql("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' ORDER BY table_name
"""))

display(sql("SELECT * FROM transactions LIMIT 5"))
display(sql("SELECT * FROM merchants LIMIT 5"))

/var/folders/8q/ww9wlcg946l4mbz6q4n9yylc0000gn/T/ipykernel_64585/1316314551.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,table_name
0,aisles
1,cards
2,customers
3,departments
4,fraud_test
5,fraud_train
6,hr_dataset
7,merchants
8,nw_categories
9,nw_customer_customer_demo


/var/folders/8q/ww9wlcg946l4mbz6q4n9yylc0000gn/T/ipykernel_64585/1316314551.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,txn_id,card_id,merchant_id,amount,txn_ts,status,channel
0,1,123,31,610000,2025-02-01 01:31:33,success,POS
1,2,109,13,78500,2025-02-01 02:05:06,success,online
2,3,16,12,12500,2025-02-01 03:56:04,success,P2P
3,4,106,30,31500,2025-02-01 04:50:37,success,POS
4,5,79,10,90500,2025-02-01 05:03:24,success,POS


/var/folders/8q/ww9wlcg946l4mbz6q4n9yylc0000gn/T/ipykernel_64585/1316314551.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,merchant_id,name,category,city
0,1,Korzinka,Supermarket,Toshkent
1,2,Makro,Supermarket,Nukus
2,3,Havas,Supermarket,Toshkent
3,4,Magnum,Supermarket,Samarqand
4,5,Carrefour,Supermarket,Farg'ona


# Базовый уровень

## Задача 1. Топ-5 категорий по обороту

Топ-5 категорий мерчантов по обороту. Только успешные платежи. Вывести оборот и число транзакций.

**Подсказка:** JOIN `transactions` и `merchants`, фильтр по success, `GROUP BY category`, `ORDER BY ... DESC LIMIT 5`.

In [ ]:
sql("""

SELECT category,
       COUNT(*)    AS transactions_count,
       SUM(amount) AS revenue
FROM public.merchants m
         JOIN public.transactions t ON m.merchant_id = t.merchant_id
WHERE t.status = 'success'
GROUP BY category
ORDER BY revenue DESC
LIMIT 5;

""")

## Задача 2. Оборот по городам клиентов

Оборот по городам клиентов. Только success, по убыванию.

**Подсказка:** Два JOIN: транзакция, карта, клиент. `GROUP BY c.city`.

In [ ]:
sql("""

SELECT city,
       SUM(amount) AS revenue
FROM merchants m
         JOIN public.transactions t ON m.merchant_id = t.merchant_id
WHERE t.status = 'success'
GROUP BY city
ORDER BY revenue DESC;

""")

## Задача 3. Доля отклонённых платежей

Доля отклонённых платежей: процент строк со `status = 'declined'` от всех. Одно число.

**Подсказка:** Считать declined через `CASE` и `SUM`, делить на `COUNT(*)`. Умножать на `100.0`, иначе будет целочисленное деление.

In [ ]:
sql("""

SELECT ROUND(100.0 * SUM(CASE WHEN status = 'declined' THEN 1 ELSE 0 END) / COUNT(*), 2) AS declined_percentage
FROM transactions;

""")

## Задача 4. Мерчанты Online с числом платежей больше 50

Мерчанты категории Online, у которых больше 50 успешных платежей.

**Подсказка:** `WHERE category = 'Online' AND status = 'success'`, `GROUP BY` мерчант, `HAVING COUNT(*) > 50`.

In [ ]:
sql("""

SELECT m.name,
       COUNT(*) AS transactions_count
FROM merchants m
         JOIN transactions t ON m.merchant_id = t.merchant_id
WHERE category = 'Online'
  AND status = 'success'
GROUP BY name
HAVING COUNT(*) > 50
ORDER BY transactions_count DESC;

""")

# Средний уровень

## Задача 5. Клиенты выше среднего

Клиенты, у которых оборот выше среднего оборота по всем клиентам.

**Подсказка:** Шаг 1 (CTE): оборот по клиенту. Шаг 2: сравнение со средним через подзапрос `(SELECT AVG(total) FROM ...)`.

In [ ]:
sql("""

WITH customer_revenue AS (
    SELECT
        c2.customer_id,
        c2.full_name,
        SUM(t.amount) AS total
    FROM transactions t
             JOIN cards c
                  ON t.card_id = c.card_id
             JOIN customers c2
                  ON c.customer_id = c2.customer_id
    WHERE t.status = 'success'
    GROUP BY c2.customer_id
)

SELECT
    customer_id,
    full_name,
    total
FROM customer_revenue
WHERE total > (
    SELECT AVG(total)
    FROM customer_revenue
)
ORDER BY total DESC;

""")

## Задача 6. Клиенты с высоким средним чеком

Средний чек и число транзакций по клиенту. Оставить тех, у кого средний чек больше 300 000.

**Подсказка:** `GROUP BY` клиент, `AVG(amount)`, `HAVING AVG(amount) > 300000`.

In [ ]:
sql("""

SELECT
    c2.customer_id,
    c2.full_name,
    AVG(t.amount) AS avg_check,
    COUNT(*) AS transaction_count
FROM transactions t
         JOIN cards c
              ON t.card_id = c.card_id
         JOIN customers c2
              ON c.customer_id = c2.customer_id
WHERE t.status = 'success'
GROUP BY c2.customer_id
HAVING AVG(t.amount) > 300000
ORDER BY avg_check DESC;

""")

## Задача 7. Карты без успешных платежей

Карты, по которым нет ни одной успешной транзакции.

**Подсказка:** `NOT EXISTS`: карта, для которой не существует ни одной success-транзакции. Альтернатива: `LEFT JOIN ... WHERE t.txn_id IS NULL`.

In [ ]:
sql("""


""")

## Задача 8. Самый крупный платёж

Самая крупная успешная транзакция: клиент, мерчант, сумма, дата.

**Подсказка:** Соединить четыре таблицы, `WHERE status = 'success'`, `ORDER BY amount DESC LIMIT 1`. Или подзапрос `WHERE amount = (SELECT MAX(amount) ...)`.

In [ ]:
sql("""

""")

## Задача 9. Главный мерчант клиента

Мерчант с наибольшей суммой трат для каждого клиента и сама сумма.

**Подсказка:** `ROW_NUMBER() OVER (PARTITION BY customer ORDER BY SUM(amount) DESC)`, затем оставить `rn = 1`.

In [ ]:
sql("""

""")

## Задача 10. Нарастающий итог по карте

Нарастающий итог трат по карте №9 по времени.

**Подсказка:** Оконная функция без `GROUP BY`: `SUM(amount) OVER (ORDER BY txn_ts)`. Каждая строка это сумма всех предыдущих.

In [ ]:
sql("""

""")

## Задача 11. Снижение оборота в мае

Клиенты, у которых оборот в мае ниже апрельского более чем на 30%.

**Подсказка:** Шаг 1: оборот по клиенту по месяцам (`date_trunc('month', txn_ts)`). Шаг 2: `LAG` для предыдущего месяца. Шаг 3: оставить май, где `total < prev * 0.7`.

In [ ]:
sql("""

""")

## OPTIONAL

## Топ-3 мерчанта в каждой категории

Топ-3 мерчанта по обороту внутри каждой категории.

**Подсказка:** `ROW_NUMBER()` или `RANK()` `OVER (PARTITION BY category ORDER BY SUM(amount) DESC)`, оставить `rn <= 3`.

In [ ]:
sql("""

""")

##  RFM по клиентам

Для каждого клиента: Recency (дней с последней покупки до 2025-06-01), Frequency (число успешных платежей), Monetary (оборот). Разбить каждую метрику на 4 группы через `NTILE(4)`.

**Подсказка:** Сначала CTE с тремя метриками. Recency: `DATE '2025-06-01' - MAX(txn_ts)::date`. Затем три `NTILE(4) OVER (ORDER BY ...)`.

In [ ]:
sql("""

""")

## Оборот по месяцам

Оборот всей системы по месяцам и процент прироста к предыдущему месяцу.

**Подсказка:** CTE: оборот по `date_trunc('month', txn_ts)`. Затем `LAG(turnover) OVER (ORDER BY ym)` и формула прироста.

In [ ]:
sql("""

""")

Закрыть соединение в конце работы:

In [ ]:
conn.close()
print('Соединение закрыто')